# Module 4 — Evaluation (instrument the agent, then score it)

In Module 2 you deployed the agent and saw its traces in CloudWatch. Traces tell you *what*
the agent did — this module tells you *how well* it did it, with **AgentCore Evaluation**.

The catch: the built-in evaluators expect traces in a specific format. Your Claude Agent SDK
agent doesn't emit that format out of the box. So first you'll use a **Claude Code Skill**
(`ac-evaluation-transform`) to instrument the agent — you'll watch it rewrite the agent's
code — then deploy, evaluate, and read the scores.

The flow:
1. **Instrument** — run the skill in Claude Code; see the agent code change (`git diff`).
2. **Deploy** the transformed agent to a new runtime (`aatransformed`).
3. **Invoke** it a few times to generate traces.
4. **On-demand eval** — `agentcore run eval` gives instant scores from all 9 evaluators.
5. **Online eval** — configure continuous, automatic scoring at 100% sampling.
6. Invoke again, wait ~20 min, then read the online results.

## Prerequisites

- **Module 0 complete** — the `student_analytics` Athena database + S3 bucket exist.
- **Claude Code CLI installed locally** — the skill runs inside Claude Code.
  Install: `npm install -g @anthropic-ai/claude-code` (needs Node.js 18+).
- **AWS credentials** with Bedrock, Athena, and CloudWatch access.
- **CDK bootstrapped** in your region: `npx cdk bootstrap aws://<account>/<region>`.

## Setup

### Python environment

Make sure you followed Module 0's setup (`uv sync` from the project root + select the `.venv` kernel).

### Install Node.js + AgentCore CLI

In [ ]:
!bash setup.sh

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")
print("Account:", acct, "| Region:", region)

---

## Step 1 — Instrument the agent with the `ac-evaluation-transform` Skill

This module ships the **same, un-instrumented agent from Module 2**. Confirm that first —
there is no `observability.py`, and the entrypoint is the plain streaming version:

In [ ]:
bundle = "analytics_agent"
print("BEFORE the skill runs:")
print(f"  observability.py exists? {os.path.isfile(f'{bundle}/observability.py')}  (expected: False)")
with open(f"{bundle}/agent_agentcore.py") as f:
    src = f.read()
print(f"  entrypoint signature: " + next(l.strip() for l in src.splitlines() if 'async def invoke' in l))

### What the skill does

The `ac-evaluation-transform` skill teaches Claude Code to add OTEL instrumentation so the
agent's traces match what AgentCore's built-in evaluators parse. It produces:

- **`observability.py`** (new) — emits the Strands-compatible span tree
  (`invoke_agent → execute_event_loop_cycle → chat → tool.*`) plus structured I/O-summary logs.
  It reuses the runtime's global ADOT providers (inside a container they're set-once — building
  your own is dead code) and degrades to a no-op locally.
- **`agent.py`** — registers `PreToolUse` / `PostToolUse` hooks in `build_agent_options()`.
- **`agent_agentcore.py`** — streams through `observability.run_instrumented()`; entrypoint gains
  a 2-arg `context` param for session-level scoring.
- **`Dockerfile`** — disables the `openinference` auto-instrumentor (it emits a competing span tree).

### Install the skill

In a **terminal**, install the skill into Claude Code (one time):

```bash
claude install-skill https://github.com/aws-samples/sample-AgentCore-make-Claude_Agent_SDK-production_ready/tree/main/skills/ac-evaluation-transform
```

### Run the skill to transform the agent

From this module's `analytics_agent/` directory, launch Claude Code with a prompt that triggers it:

```bash
cd advanced/agentic-analytics/module-4-evaluation/analytics_agent
claude "This agent will be deployed to Amazon Bedrock AgentCore Runtime with enableOtel: true. \
  Use the ac-evaluation-transform skill to instrument it so all 9 AgentCore built-in evaluators \
  can score its traces. Target: deployed AgentCore Runtime — reuse the global ADOT providers, do \
  NOT build your own TracerProvider/exporters. Keep all existing tools and prompts unchanged."
```

Claude Code will read the skill, inspect the agent, and edit the files. **Wait for it to finish**,
then run the next cell to see exactly what changed.

In [ ]:
# Show what the skill changed — git makes the transformation visible.
import subprocess

print("=== New / modified files in the agent bundle ===")
print(subprocess.run(["git", "status", "--short", bundle],
                     capture_output=True, text=True).stdout or "(no changes detected)")

print("=== Lines added/removed per file ===")
print(subprocess.run(["git", "diff", "--stat", bundle],
                     capture_output=True, text=True).stdout or "(no tracked-file changes)")

In [ ]:
# Confirm the transformation is complete and correct.
print("AFTER the skill runs:")
ok = os.path.isfile(f"{bundle}/observability.py")
print(f"  observability.py exists? {ok}  (expected: True)")

if ok:
    obs = open(f"{bundle}/observability.py").read()
    entry = open(f"{bundle}/agent_agentcore.py").read()
    docker = open(f"{bundle}/Dockerfile").read()
    for cond, label in [
        ("AGENT_OBSERVABILITY_ENABLED" in obs, "no-op gate (safe when disabled)"),
        ("get_logger_provider" in obs, "reuses global ADOT providers"),
        ("invoke_agent" in obs, "Strands-compatible span tree"),
        ("post_tool_use_hook" in obs, "per-tool I/O-summary hooks"),
        ("async def invoke(payload" in entry and "context" in entry, "2-arg context entrypoint"),
        ("claude_agent_sdk" in docker and "DISABLED_INSTRUMENTATIONS" in docker, "openinference disabled"),
    ]:
        print(f"  {'✅' if cond else '❌'} {label}")
else:
    print("\n⚠️  The skill hasn't run yet (or didn't complete). Options:")
    print("    - Re-run the skill in Claude Code (see the cell above), OR")
    print("    - Use the reference: cp skill_output_reference/*.py \\")
    print("        skill_output_reference/Dockerfile analytics_agent/")
    print("      (a known-good transformation, so you can proceed without Claude Code CLI).")

> **No Claude Code CLI?** This module ships a known-good transformation under
> `skill_output_reference/`. Copy those four files into `analytics_agent/` and continue:
> ```bash
> cp skill_output_reference/*.py skill_output_reference/Dockerfile analytics_agent/
> ```
> The point of Step 1 is to *see* the skill do this for you — the reference is just a fallback.

---

## Step 2 — Deploy the instrumented agent

This deploys to a **new** AgentCore project (`aatransformed`) — independent from Module 2's runtime,
so you can tell the two apart. The CDK stack carries the Athena/Glue/S3 permissions and `enableOtel: true`.

In [ ]:
import json
targets = [{"name": "default", "description": "eval target",
            "account": acct, "region": region}]
with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)
print("wrote agentcore/aws-targets.json →", acct, region)

In [ ]:
!agentcore deploy -y

In [ ]:
!agentcore status

---

## Step 3 — Invoke the agent (generate traces)

Send a few questions. Each invocation emits the Strands-format span tree + I/O-summary logs
the evaluators will score. Use session ids ≥ 33 characters (a runtime requirement).

In [ ]:
!agentcore invoke '{"prompt": "How many distinct students are enrolled in total?"}' \
    --session-id agentic-analytics-m4-eval-session-001

In [ ]:
!agentcore invoke '{"prompt": "What is the average tuition per student for Fall 2024?"}' \
    --session-id agentic-analytics-m4-eval-session-001

In [ ]:
!agentcore invoke '{"prompt": "Which major has the highest enrollment? Show me the top 5."}' \
    --session-id agentic-analytics-m4-eval-session-002

Confirm traces are flowing (allow 2–3 minutes for indexing):

In [ ]:
!agentcore traces list --runtime analytics --since 1h

---

## Step 4 — On-demand evaluation (instant results)

Run all **9 built-in evaluators** against the traces you just generated — no waiting.

| Evaluator | Measures |
|-----------|----------|
| `Builtin.Helpfulness` | Does the response address the user's need? |
| `Builtin.Faithfulness` | Is it grounded in the tool outputs? |
| `Builtin.Correctness` | Is the factual content accurate? |
| `Builtin.Coherence` | Is it logically structured? |
| `Builtin.Conciseness` | Is it appropriately brief? |
| `Builtin.Harmfulness` | Free of harmful content? (1.0 = safe) |
| `Builtin.InstructionFollowing` | Did it follow the system prompt? |
| `Builtin.GoalSuccessRate` | Did the session achieve its goal? (session-level) |
| `Builtin.ToolSelectionAccuracy` | Did it pick the right tools? |

In [ ]:
!agentcore run eval --runtime analytics \
    --evaluator Builtin.Helpfulness Builtin.Faithfulness Builtin.Correctness \
               Builtin.Coherence Builtin.Conciseness Builtin.Harmfulness \
               Builtin.InstructionFollowing Builtin.GoalSuccessRate \
               Builtin.ToolSelectionAccuracy \
    --session-id agentic-analytics-m4-eval-session-001 \
    --days 1

You should see a numeric score (0.0–1.0) for each evaluator. **This is the proof the
instrumentation worked** — if the skill hadn't transformed the agent, you'd get
`AgentSpanMappingException` or `LogEventMissingException` instead of scores.

Score the second session too:

In [ ]:
!agentcore run eval --runtime analytics \
    --evaluator Builtin.Helpfulness Builtin.Faithfulness Builtin.Correctness \
               Builtin.Coherence Builtin.Conciseness Builtin.Harmfulness \
               Builtin.InstructionFollowing Builtin.GoalSuccessRate \
               Builtin.ToolSelectionAccuracy \
    --session-id agentic-analytics-m4-eval-session-002 \
    --days 1

**Reading the scores:** 0.0–1.0, higher is better (for `Harmfulness`, 1.0 = safe).
`GoalSuccessRate` is session-level. `Conciseness` often scores lower for agents that explain
their reasoning — that's a comment on content, not on the instrumentation.

---

## Step 5 — Configure online evaluation (continuous monitoring)

On-demand eval is for spot-checks. In production you want **every invocation** scored
automatically. Online evaluation runs the evaluators on a sampling rate (100% here), stores
the results, and can be paused/resumed anytime.

In [ ]:
!agentcore add online-eval \
    --name eval-all-builtin \
    --runtime analytics \
    --evaluator Builtin.Helpfulness Builtin.Faithfulness Builtin.Correctness \
               Builtin.Coherence Builtin.Conciseness Builtin.Harmfulness \
               Builtin.InstructionFollowing Builtin.GoalSuccessRate \
               Builtin.ToolSelectionAccuracy \
    --sampling-rate 100 \
    --enable-on-create

Deploy the updated config (the online eval config is provisioned by the CDK stack):

In [ ]:
!agentcore deploy -y

In [ ]:
!agentcore status

---

## Step 6 — Invoke, then read the online results

These invocations are scored **automatically** by the online eval config.

In [ ]:
!agentcore invoke '{"prompt": "What percentage of students have outstanding balances over $5000?"}' \
    --session-id agentic-analytics-m4-online-session-01

In [ ]:
!agentcore invoke '{"prompt": "Show me the scholarship distribution by department for Spring 2024."}' \
    --session-id agentic-analytics-m4-online-session-02

### Wait ~20 minutes

Online results take ~15–20 minutes to appear: the agent emits spans/logs → CloudWatch indexes
them (~2–3 min) → the online eval service picks up the new traces and runs all 9 evaluators.

While you wait, you can re-run the Step 4 on-demand eval on these new sessions to see the same
scores immediately, or open the CloudWatch GenAI Observability console:

```
https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/sessions
```

In [ ]:
# Run this after ~20 minutes.
!agentcore evals history

You can also view the scores in the **AgentCore console**: Amazon Bedrock → AgentCore →
Agent Runtimes → the `aatransformed` runtime → **Evaluation** tab.

Pause or resume online evaluation anytime:

In [ ]:
# !agentcore pause online-eval eval-all-builtin
# !agentcore resume online-eval eval-all-builtin

---

## Cleanup

Tear down the runtime + online eval config when you're done:

In [ ]:
# !agentcore remove -y

## Recap

- **The skill did the hard part.** `ac-evaluation-transform` rewrote the agent to emit
  evaluator-compatible traces — you watched it change `agent.py`, `agent_agentcore.py`,
  `Dockerfile`, and add `observability.py`. No hand-written OTEL.
- **On-demand eval** (`agentcore run eval`) gives instant scores on any session or trace.
- **Online eval** (`agentcore add online-eval` + deploy) scores every invocation automatically.
- **9 built-in evaluators** returned numeric scores — proof the instrumentation is correct.

That completes the Agentic Analytics track: **set up data (M0) → build the agent (M1) →
deploy & observe (M2) → ask good follow-ups (M3) → measure quality (M4).**